In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [7]:
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

# Define the paths to the prediction files
prediction_files = {
    "D-DGCN": "/content/drive/MyDrive/hust/nlp/outputs/ddgcn_predictions.csv",
    "RAG Multi-Agent": "/content/drive/MyDrive/hust/nlp/outputs/rag_multi_agent_predictions.csv",
    "RoBERTa": "/content/drive/MyDrive/hust/nlp/outputs/roberta_predictions.csv",
    "SVM": "/content/drive/MyDrive/hust/nlp/outputs/svm_predictions.csv"
}

# Function to convert 4 binary labels to 16-class MBTI string
def binary_labels_to_mbti(ie, sn, tf, pj):
    mbti_type = ""
    mbti_type += 'I' if ie == 0 else 'E'
    mbti_type += 'S' if sn == 0 else 'N'
    mbti_type += 'T' if tf == 0 else 'F'
    mbti_type += 'P' if pj == 0 else 'J' # Assuming pj corresponds to P/J where P=0, J=1
    return mbti_type

# Initialize a dictionary to store all results
all_results = {}

# Iterate through each model's prediction file
for model_name, file_path in prediction_files.items():
    print(f"Processing {model_name}...")
    try:
        df = pd.read_csv(file_path)

        # Ensure all required binary label columns exist and are numeric
        required_cols = [
            'y_true_label_ie', 'y_pred_label_ie',
            'y_true_label_sn', 'y_pred_label_sn',
            'y_true_label_tf', 'y_pred_label_tf',
            'y_true_label_jp', 'y_pred_label_jp'
        ]

        if not all(col in df.columns for col in required_cols):
            print(f"Error: One or more required columns not found in {file_path}")
            print(f"Expected: {required_cols}")
            print(f"Found: {df.columns.tolist()}")
            continue

        # Convert columns to integers (binary labels)
        for col in required_cols:
            df[col] = df[col].astype(int)

        # Reconstruct 16-class MBTI types from binary labels for overall metrics
        y_true_16_class = [binary_labels_to_mbti(i, s, t, j)
                           for i, s, t, j in zip(df['y_true_label_ie'], df['y_true_label_sn'],
                                                 df['y_true_label_tf'], df['y_true_label_jp'])]
        y_pred_16_class = [binary_labels_to_mbti(i, s, t, j)
                           for i, s, t, j in zip(df['y_pred_label_ie'], df['y_pred_label_sn'],
                                                 df['y_pred_label_tf'], df['y_pred_label_jp'])]

        # Overall Metrics (16-class classification)
        overall_accuracy = accuracy_score(y_true_16_class, y_pred_16_class)
        overall_macro_f1 = f1_score(y_true_16_class, y_pred_16_class, average='macro', zero_division=0)

        # Per-axis Metrics (4 binary classifications)
        f1_ie = f1_score(df['y_true_label_ie'], df['y_pred_label_ie'], average='macro', zero_division=0)
        f1_sn = f1_score(df['y_true_label_sn'], df['y_pred_label_sn'], average='macro', zero_division=0)
        f1_tf = f1_score(df['y_true_label_tf'], df['y_pred_label_tf'], average='macro', zero_division=0)
        f1_pj = f1_score(df['y_true_label_jp'], df['y_pred_label_jp'], average='macro', zero_division=0) # Using 'jp' for P/J

        all_results[model_name] = {
            "Overall Accuracy": overall_accuracy,
            "Overall Macro F1": overall_macro_f1,
            "F1 (I/E)": f1_ie,
            "F1 (S/N)": f1_sn,
            "F1 (T/F)": f1_tf,
            "F1 (P/J)": f1_pj
        }

    except FileNotFoundError:
        print(f"Error: File not found at {file_path}")
    except Exception as e:
        print(f"An error occurred while processing {model_name}: {e}")

# Create a DataFrame from the results for easy comparison
results_df = pd.DataFrame.from_dict(all_results, orient='index')
results_df.index.name = 'Model'

print("\n--- MBTI Classification Results ---")
display(results_df.round(4))


Processing D-DGCN...
Processing RAG Multi-Agent...
Processing RoBERTa...
Processing SVM...

--- MBTI Classification Results ---


,Overall Accuracy,Overall Macro F1,F1 (I/E),F1 (S/N),F1 (T/F),F1 (P/J)
Model,,,,,,
D-DGCN,0.4911,0.3659,0.7244,0.7242,0.8262,0.7240
RAG Multi-Agent,0.0300,0.0042,0.2063,0.4444,0.3151,0.3007
RoBERTa,0.4329,0.2906,0.7384,0.7021,0.8069,0.7165
SVM,0.4375,0.2394,0.7011,0.6165,0.8163,0.6839
